# PiTheta Cloud Train Notebook

One-click orchestration for `build -> train -> eval -> export`.

Before running:
1. Open notebook from repository root.
2. Ensure datasets exist in `data/` and model path `deberta-v3-base/` is available.
3. Run in a Python environment with GPU support if available.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
print('ROOT:', ROOT)

def run_cmd(argv: list[str], *, cwd: Path = ROOT) -> str:
    print('\n$ ' + ' '.join(argv))
    proc = subprocess.run(
        argv,
        cwd=str(cwd),
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace',
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr)
        raise RuntimeError(f'command failed with code {proc.returncode}')
    return proc.stdout

def extract_json_blob(text: str) -> dict:
    start = text.find('{')
    end = text.rfind('}')
    if start < 0 or end < start:
        raise ValueError('JSON payload not found in command output')
    return json.loads(text[start:end+1])


In [ ]:
# Optional dependency install for cloud environments.
INSTALL_DEPS = False
if INSTALL_DEPS:
    run_cmd([sys.executable, '-m', 'pip', 'install', '-r', 'cloud/pitheta/requirements-train.lock'])
else:
    print('INSTALL_DEPS=False (skipped)')


In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('cloud_%Y%m%dT%H%M%SZ')
DATA_DIR = ROOT / 'artifacts' / 'pitheta_data' / RUN_ID
TRAIN_DIR = ROOT / 'artifacts' / 'pitheta_train' / RUN_ID
EVAL_DIR = ROOT / 'artifacts' / 'pitheta_eval' / RUN_ID
EXPORT_DIR = ROOT / 'omega' / 'models' / 'pitheta' / RUN_ID
BASELINE_PATH = ROOT / 'artifacts' / 'pitheta_eval' / f'baseline_{RUN_ID}.json'

print('RUN_ID:', RUN_ID)
print('DATA_DIR:', DATA_DIR)
print('TRAIN_DIR:', TRAIN_DIR)
print('EVAL_DIR:', EVAL_DIR)
print('EXPORT_DIR:', EXPORT_DIR)


In [ ]:
def run_full_pipeline() -> dict:
    # 1) Build dataset artifacts.
    run_cmd([
        sys.executable,
        'scripts/build_pitheta_dataset.py',
        '--registry', 'config/pitheta_dataset_registry.yml',
        '--output-dir', str(DATA_DIR),
        '--seed', '41',
        '--strict',
    ])

    # 2) Baseline snapshot with pi0 for regression compare.
    baseline_stdout = run_cmd([
        sys.executable,
        'scripts/run_eval.py',
        '--projector-mode', 'pi0',
        '--whitebox-max-samples', '200',
        '--run-deepset',
        '--deepset-benchmark-root', 'data/deepset-prompt-injections',
        '--deepset-split', 'test',
        '--deepset-mode', 'full',
        '--deepset-seed', '41',
    ])
    baseline = extract_json_blob(baseline_stdout)
    BASELINE_PATH.parent.mkdir(parents=True, exist_ok=True)
    BASELINE_PATH.write_text(json.dumps(baseline, ensure_ascii=True, indent=2), encoding='utf-8')
    print('Baseline report:', BASELINE_PATH)

    # 3) Train PiTheta LoRA.
    run_cmd([
        sys.executable,
        'scripts/train_pitheta_lora.py',
        '--train-config', 'config/pitheta_train.yml',
        '--data-dir', str(DATA_DIR),
        '--output-dir', str(TRAIN_DIR),
    ])

    # 4) Evaluate against strict gates.
    run_cmd([
        sys.executable,
        'scripts/eval_pitheta.py',
        '--checkpoint', str(TRAIN_DIR / 'best'),
        '--data-dir', str(DATA_DIR),
        '--baseline-report', str(BASELINE_PATH),
        '--output-dir', str(EVAL_DIR),
        '--strict-gates',
    ])

    # 5) Export runtime adapter.
    run_cmd([
        sys.executable,
        'scripts/export_pitheta_adapter.py',
        '--checkpoint-dir', str(TRAIN_DIR / 'best'),
        '--export-dir', str(EXPORT_DIR),
        '--model-manifest', str(TRAIN_DIR / 'model_manifest.json'),
    ])

    summary = {
        'run_id': RUN_ID,
        'data_dir': str(DATA_DIR),
        'train_dir': str(TRAIN_DIR),
        'eval_dir': str(EVAL_DIR),
        'export_dir': str(EXPORT_DIR),
        'baseline_report': str(BASELINE_PATH),
        'gate_report': str(EVAL_DIR / 'gate_report.json'),
    }
    print('\nPipeline completed:')
    print(json.dumps(summary, ensure_ascii=True, indent=2))
    return summary

summary = run_full_pipeline()